## Encoding

#### 컴퓨터가 이해할 수 없는 **범주형(Categorical) 데이터를 수치형(Numerical) 데이터로 변환**하는 전처리 과정이다.
> 대부분의 ML 알고리즘은 수치 입력만 처리할 수 있으므로, 문자열·범주형 피처를 적절한 방식으로 숫자화해야 한다.
> 인코딩 방식에 따라 모델 성능과 해석력이 크게 달라질 수 있으므로, 데이터 특성과 모델 유형에 맞는 기법을 선택하는 것이 중요하다.

![](https://velog.velcdn.com/images/newnew_daddy/post/0874e71f-df1a-4b89-a967-e0de5ff748a3/image.png)


| 구분 | Label Encoding | One-Hot Encoding | Ordinal Encoding | Target Encoding |
|---|---|---|---|---|
| **변환 방식** | 범주 → 정수 (0, 1, 2, …) | 범주 → 이진 벡터 (0/1) | 범주 → 순서 반영 정수 | 범주 → 타겟 변수의 평균값 |
| **순서 의미** | 없음 (임의 할당) | 없음 | **있음** (순서 반영) | 없음 (타겟 기반) |
| **차원 변화** | 변화 없음 | 범주 수만큼 증가 | 변화 없음 | 변화 없음 |
| **적합 모델** | 트리 기반 (RF, XGBoost 등) | 회귀, SVM, 신경망 등 | 트리 기반 / 순서형 데이터 | 고카디널리티 범주형 피처 |
| **장점** | 단순·빠른 구현 | 범주 간 독립성 보장 | 순서 정보 보존 | 고유값이 많아도 차원 증가 없음 |
| **단점** | 크기 관계 오해 가능 | 고카디널리티 시 차원 폭증 | 순서 정의가 필요 | 과적합에 취약 (CV·정규화 필수) |
| **예시** | ["a","b","c"] → [0,1,2] | ["a","b","c"] → [[1,0,0],[0,1,0],[0,0,1]] | ["S","M","L"] → [0,1,2] | ["a","b"] → [0.75, 0.33] |

ref: https://velog.io/@seungwoong12/encoding

#### 1. Label Encoding
- 범주형 변수를 정수로 변환합니다. 범주 간 순서가 의미 없을 때 사용합니다.
- 주로 트리 기반 모델에서 사용되며, 단순하고 빠르게 구현할 수 있습니다.
- ["a", "b", "c"] -> [0, 1, 2]

In [9]:
import pandas as pd

df = pd.read_csv("./dataset/encoding.csv")

df.head()

,color,size,target
0,red,S,10
1,blue,M,13
2,green,L,8
3,blue,M,4
4,green,S,11


In [10]:
df.dtypes

color       str
size        str
target    int64
dtype: object

In [11]:
## Label Encoding -> 각 카테고리 값을 숫자형으로 변환

df['color_label'] = df['color'].astype('category')

print(df)

    color size  target color_label
0     red    S      10         red
1    blue    M      13        blue
2   green    L       8       green
3    blue    M       4        blue
4   green    S      11       green
5     red    L      14         red
6     red    S       9         red
7    blue    M      10        blue
8     red    S       2         red
9    blue    M       8        blue
10  green    L       7       green
11   blue    M      15        blue
12  green    S      12       green
13    red    L      10         red
14    red    S       3         red
15   blue    M       1        blue


In [12]:
df.dtypes

color               str
size                str
target            int64
color_label    category
dtype: object

In [13]:
print(df['color_label'].cat.categories)

df['color_label'] = df['color_label'].cat.codes

Index(['blue', 'green', 'red'], dtype='str')


In [14]:
df.head()

,color,size,target,color_label
0,red,S,10,2
1,blue,M,13,0
2,green,L,8,1
3,blue,M,4,0
4,green,S,11,1


In [15]:
## factorize 함수 사용 -> 2가지 값을 return 한다.

arr, ind = df['color'].factorize()

arr, ind

(array([0, 1, 2, 1, 2, 0, 0, 1, 0, 1, 2, 1, 2, 0, 0, 1]),
 Index(['red', 'blue', 'green'], dtype='str'))

In [16]:
## factorize 함수 사용

df['factorize'] = df['color'].factorize()[0]

df

,color,size,target,color_label,factorize
0,red,S,10,2,0
1,blue,M,13,0,1
2,green,L,8,1,2
3,blue,M,4,0,1
4,green,S,11,1,2
5,red,L,14,2,0
6,red,S,9,2,0
7,blue,M,10,0,1
8,red,S,2,2,0
9,blue,M,8,0,1


#### fit, transform, fit_transform 의 차이점

1. fit
- 모델이나 전처리 객체가 데이터를 학습하는 과정입니다. fit은 주어진 데이터로 학습을 진행하며, 주로 매개변수나 통계를 계산하여 객체에 저장합니다. 하지만 데이터를 변환하지는 않습니다.

2. transform
- 이미 학습된 객체가(즉, fit이 수행된 후) 데이터를 변환하는 작업을 합니다. fit에서 계산된 정보(예: 평균, 분산)를 사용하여 입력 데이터를 변환합니다.

3. fit_transform
- fit과 transform을 한 번에 수행하는 메소드입니다. 데이터를 학습하고, 그 데이터를 바로 변환합니다. 주로 전처리 과정에서 많이 사용됩니다.

#### INPUT 될 수 있는 데이터의 형상
- [dataframe].shape 찍었을 때, (행, 열) 이런식으로 나오는 2D 배열 형식이 유효
  - Pandas DataFrame : 바로 사용 가능
  - Pandas Series
    - `to_frame()` 함수를 사용하여 변환
    - `values` 메소드를 통해 numpy 배열로 변환 후 `reshape(-1,1)` 로 다시 재배열


In [17]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(df['color'])

df['color_label_sklearn'] = le.transform(df['color'])

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['color_label_sklearn'] = le.fit_transform(df['color'])
print(df)

    color size  target  color_label  factorize  color_label_sklearn
0     red    S      10            2          0                    2
1    blue    M      13            0          1                    0
2   green    L       8            1          2                    1
3    blue    M       4            0          1                    0
4   green    S      11            1          2                    1
5     red    L      14            2          0                    2
6     red    S       9            2          0                    2
7    blue    M      10            0          1                    0
8     red    S       2            2          0                    2
9    blue    M       8            0          1                    0
10  green    L       7            1          2                    1
11   blue    M      15            0          1                    0
12  green    S      12            1          2                    1
13    red    L      10            2          0  

#### 2. One-Hot Encoding
- 범주형 변수를 이진 변수의 배열로 변환합니다. 범주 간 순서가 없는 경우 사용합니다.
- 각 범주가 서로 독립적일 때 유용하며, 회귀나 SVM 모델에서 많이 사용됩니다.
- ["a", "b", "c"] -> [[1, 0, 0], [0, 1, 0], [0, 0, 1]]

#### Label Enc vs One-hot Enc

![](https://velog.velcdn.com/images/newnew_daddy/post/a0f93c4f-0f04-4d93-951b-9b2041638cfa/image.png)


In [19]:
## One-Hot Encoding -> 각 카테고리 값을 이진 변수(dummy 변수)로 변환

# df_one_hot = pd.get_dummies(df, columns=['color'])
df_one_hot = pd.get_dummies(df, columns=['color', 'size'])
print(df_one_hot)

    target  color_label  factorize  color_label_sklearn  color_blue  \
0       10            2          0                    2       False   
1       13            0          1                    0        True   
2        8            1          2                    1       False   
3        4            0          1                    0        True   
4       11            1          2                    1       False   
5       14            2          0                    2       False   
6        9            2          0                    2       False   
7       10            0          1                    0        True   
8        2            2          0                    2       False   
9        8            0          1                    0        True   
10       7            1          2                    1       False   
11      15            0          1                    0        True   
12      12            1          2                    1       False   
13    

In [20]:
df.replace(to_replace=None, value=None)

TypeError: 'regex' must be a string or a compiled regular expression or a list or dict of strings or regular expressions, you passed a 'bool'

In [21]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()
encoded = ohe.fit_transform(df[['color', 'size']])
df_one_hot_sklearn = pd.DataFrame(encoded.toarray(), columns=ohe.get_feature_names_out(['color', 'size']))
print(df_one_hot_sklearn)

    color_blue  color_green  color_red  size_L  size_M  size_S
0          0.0          0.0        1.0     0.0     0.0     1.0
1          1.0          0.0        0.0     0.0     1.0     0.0
2          0.0          1.0        0.0     1.0     0.0     0.0
3          1.0          0.0        0.0     0.0     1.0     0.0
4          0.0          1.0        0.0     0.0     0.0     1.0
5          0.0          0.0        1.0     1.0     0.0     0.0
6          0.0          0.0        1.0     0.0     0.0     1.0
7          1.0          0.0        0.0     0.0     1.0     0.0
8          0.0          0.0        1.0     0.0     0.0     1.0
9          1.0          0.0        0.0     0.0     1.0     0.0
10         0.0          1.0        0.0     1.0     0.0     0.0
11         1.0          0.0        0.0     0.0     1.0     0.0
12         0.0          1.0        0.0     0.0     0.0     1.0
13         0.0          0.0        1.0     1.0     0.0     0.0
14         0.0          0.0        1.0     0.0     0.0 

#### 3. Ordinal Encoding
- 범주형 변수를 정수로 변환합니다. 범주 간 순서가 의미 있을 때 사용합니다.
- 범주의 순서가 중요한 경우에 유용하며, 특히 순위형 데이터에서 많이 사용됩니다.
- ["low", "medium", "high"] -> [0, 1, 2]

In [22]:
## Ordinal Encoding -> 순서가 있는 카테고리 값을 숫자형으로 변환

size_mapping = {'S': 1, 'M': 2, 'L': 3}
df['size_ordinal'] = df['size'].map(size_mapping)
print(df)

    color size  target  color_label  factorize  color_label_sklearn  \
0     red    S      10            2          0                    2   
1    blue    M      13            0          1                    0   
2   green    L       8            1          2                    1   
3    blue    M       4            0          1                    0   
4   green    S      11            1          2                    1   
5     red    L      14            2          0                    2   
6     red    S       9            2          0                    2   
7    blue    M      10            0          1                    0   
8     red    S       2            2          0                    2   
9    blue    M       8            0          1                    0   
10  green    L       7            1          2                    1   
11   blue    M      15            0          1                    0   
12  green    S      12            1          2                    1   
13    

In [23]:
from sklearn.preprocessing import OrdinalEncoder

size_mapping = {'S': 1, 'M': 2, 'L': 3}
ordinal_encoder = OrdinalEncoder(categories=[['S', 'M', 'L']])
df['size_ordinal_sklearn'] = ordinal_encoder.fit_transform(df[['size']])
print(df)

    color size  target  color_label  factorize  color_label_sklearn  \
0     red    S      10            2          0                    2   
1    blue    M      13            0          1                    0   
2   green    L       8            1          2                    1   
3    blue    M       4            0          1                    0   
4   green    S      11            1          2                    1   
5     red    L      14            2          0                    2   
6     red    S       9            2          0                    2   
7    blue    M      10            0          1                    0   
8     red    S       2            2          0                    2   
9    blue    M       8            0          1                    0   
10  green    L       7            1          2                    1   
11   blue    M      15            0          1                    0   
12  green    S      12            1          2                    1   
13    

#### 4. Target Encoding
- 범주형 데이터를 종속변수(target)값의 크기와 비례한 숫자로 변환되는 기법
- 범주형의 독립변수들이 종속변수의 값들과 상관관계가 있음을 가정하여 동작하는 기법
- Label Encoding과 유사하나 label과 target이 직접적으로 연관이 있다는 점이 차이

- 장점
  - 데이터 셋의 양에 영향을 받지 않아 빠른 학습이 가능
- 단점  
  - 과적합(Overfitting) 문제에 매우 취약하여 교차검증(Cross Validation)이나 정규화(Regularization) 같은 후속 조치들이 거의 필수적

In [24]:
df.head()

,color,size,target,color_label,factorize,color_label_sklearn,size_ordinal,size_ordinal_sklearn
0,red,S,10,2,0,2,1,0.0
1,blue,M,13,0,1,0,2,1.0
2,green,L,8,1,2,1,3,2.0
3,blue,M,4,0,1,0,2,1.0
4,green,S,11,1,2,1,1,0.0


In [26]:
df.groupby('color')['target'].mean()


color
blue     8.5
green    9.5
red      8.0
Name: target, dtype: float64

In [27]:
## Target Encoding -> 각 카테고리 값에 대해 목표 변수의 평균을 사용

target_mean = df.groupby('color')['target'].mean()
df['color_target'] = df['color'].map(target_mean)

print(df)
target_mean

    color size  target  color_label  factorize  color_label_sklearn  \
0     red    S      10            2          0                    2   
1    blue    M      13            0          1                    0   
2   green    L       8            1          2                    1   
3    blue    M       4            0          1                    0   
4   green    S      11            1          2                    1   
5     red    L      14            2          0                    2   
6     red    S       9            2          0                    2   
7    blue    M      10            0          1                    0   
8     red    S       2            2          0                    2   
9    blue    M       8            0          1                    0   
10  green    L       7            1          2                    1   
11   blue    M      15            0          1                    0   
12  green    S      12            1          2                    1   
13    

color
blue     8.5
green    9.5
red      8.0
Name: target, dtype: float64

In [28]:
df.head()

,color,size,target,color_label,factorize,color_label_sklearn,size_ordinal,size_ordinal_sklearn,color_target
0,red,S,10,2,0,2,1,0.0,8.0
1,blue,M,13,0,1,0,2,1.0,8.5
2,green,L,8,1,2,1,3,2.0,9.5
3,blue,M,4,0,1,0,2,1.0,8.5
4,green,S,11,1,2,1,1,0.0,9.5


In [36]:
from sklearn.preprocessing import TargetEncoder

te = TargetEncoder(smooth=0, target_type='continuous', random_state=42)

te.fit_transform(df['color'].values.reshape(-1, 1), df.target)

df['color_target'] = te.transform(df['color'].values.reshape(-1,1))

df

,color,size,target,color_label,factorize,color_label_sklearn,size_ordinal,size_ordinal_sklearn,color_target
0,red,S,10,2,0,2,1,0.0,8.0
1,blue,M,13,0,1,0,2,1.0,8.5
2,green,L,8,1,2,1,3,2.0,9.5
3,blue,M,4,0,1,0,2,1.0,8.5
4,green,S,11,1,2,1,1,0.0,9.5
5,red,L,14,2,0,2,3,2.0,8.0
6,red,S,9,2,0,2,1,0.0,8.0
7,blue,M,10,0,1,0,2,1.0,8.5
8,red,S,2,2,0,2,1,0.0,8.0
9,blue,M,8,0,1,0,2,1.0,8.5
